# Image Recognition: Classifying Images Based on Nearby Feature Representations

**Practical Hands-On Demonstration (Google Colab Ready)**

Welcome to the companion practical notebook for the lecture **"Image Recognition: Classifying Images Based on Nearby Feature Representations."**

### What You Will Experience in This Notebook
1. **Environment Setup** — Configure Python, NumPy, Matplotlib, and Scikit-learn (runs fast on CPU—no GPU needed!).
2. **Dataset Loading** — Load Scikit-learn's classic `digits` dataset ($1,797$ handwritten digit images across classes `0–9`) and verify our lecture's 3-feature worked example.
3. **Image Visualization** — Inspect grayscale pixel matrices ($8 \times 8$) and see how color RGB tensors ($H \times W \times 3$) work numerically.
4. **Image Preprocessing** — Normalize pixel intensities to $[0, 1]$ and create a leak-free stratified Train/Test split ($75\% / 25\%$).
5. **Feature Extraction** — Build three representations: **Raw Pixel Vectors ($64$-D)**, **Compact PCA Projections ($24$-D)**, and **Handcrafted Center-Aligned + Edge Gradient (Mini-HOG) Features ($128$-D)**.
6. **Feature Vector & Feature Space Visualization** — Compare an $8 \times 8$ matrix with its $64$-D vector bar plot and project all $1,797$ images into a 2D Feature Space scatter plot to see class clusters.
7. **Distance & Similarity Calculation** — Verify both **Euclidean Distance** and **Cosine Similarity** on our lecture's worked example and real digit pairs.
8. **KNN Classification** — Fit `KNeighborsClassifier` ($K = 3$) and observe how lazy instance-based learning works.
9. **Testing with Unseen Images** — Classify held-out test images and inspect neighbor vote breakdowns.
10. **Accuracy Evaluation** — Compute overall test accuracy ($98.44\%$) and per-digit Precision, Recall, and F1-scores.
11. **Confusion Matrix & Error Inspection** — Plot the $10 \times 10$ confusion matrix and visually examine misclassified digits.
12. **Visualization of Nearest Neighbors** — Display unseen query images directly beside their top-$K$ closest training neighbors and exact distances.
13. **Experimenting with Different $K$ Values** — Sweep $K \in [1, 25]$ to visualize Overfitting vs. Optimal $K$ vs. Underfitting.
14. **Raw Pixels vs. Improved Handcrafted Features** — Run a real-world **1-pixel shift + $40\%$ dimmed lighting** stress test (`52.00%` vs. `98.00%`!).
15. **Test Your Own Custom Image & Summary** — Test a custom drawn digit out-of-the-box **or upload your own image file in Google Colab**, followed by key conclusions.

---
## 1. Environment Setup

Let's import `numpy`, `matplotlib`, `PIL`, and `scikit-learn` modules, set a fixed random seed for 100% reproducible results, and configure clean dark-friendly/high-contrast plot styling suitable for classroom projection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

# Reproducibility & clean visual defaults
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"NumPy version        : {np.__version__}")
print(f"Scikit-learn version : {sklearn.__version__}")
print("Environment ready! Running on lightweight CPU mode.")

---
## 2. Dataset Loading

We use Scikit-learn's built-in **Optical Recognition of Handwritten Digits (`load_digits`)** dataset:
- **Zero external downloads or local file paths needed**—runs anywhere instantly.
- **1,797 grayscale images** of handwritten digits from **`0` to `9`** written by dozens of different people.
- Each image is an **$8 \times 8$ grid** ($64$ pixels total), with integer ink intensities from **`0` (white background)** to **`16` (dark ink)**.
- Balanced across all 10 classes (~180 examples per digit).

In [ ]:
digits = load_digits()

images_raw = digits.images      # Shape: (1797, 8, 8) -> 2D image matrices
X_raw_16   = digits.data        # Shape: (1797, 64)   -> Flattened 1D pixel vectors
y          = digits.target      # Shape: (1797,)      -> True digit labels (0..9)
class_names = digits.target_names

print(f"Total images loaded       : {len(images_raw)}")
print(f"2D Image matrix shape     : {images_raw[0].shape} (Height x Width)")
print(f"Flattened vector shape    : {X_raw_16[0].shape} (64 features per image)")
print(f"Pixel intensity range     : [{X_raw_16.min():.1f}, {X_raw_16.max():.1f}]")
print(f"Classes                   : {class_names.tolist()}")
print(f"Samples per digit (0..9)  : {np.bincount(y).tolist()}")

---
## 3. Image Visualization: Pixels, Matrices & RGB Channels

How does a computer represent images numerically?
1. **Grayscale image ($H \times W$):** A single 2D matrix of brightness/ink numbers. Below we visualize sample digits alongside the exact $8 \times 8$ numeric matrix for `Digit 0`.
2. **Color RGB image ($H \times W \times 3$):** Three stacked 2D matrices (**Red**, **Green**, and **Blue** channels). We also construct a tiny synthetic $8 \times 8 \times 3$ color icon below so you can see all three numeric channels side by side!

In [ ]:
# Part A: Sample gallery of handwritten digits (0 through 9)
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
fig.suptitle("Sample Handwritten Digits from sklearn.datasets.load_digits (8x8 Pixels)", fontsize=12, fontweight="bold")

for digit_cls, ax in enumerate(axes.ravel()):
    sample_idx = np.where(y == digit_cls)[0][0]
    ax.imshow(images_raw[sample_idx], cmap="Blues", vmin=0, vmax=16)
    ax.set_title(f"Class: Digit {digit_cls}", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part B: Zooming into Image #0 — Human View vs. Computer's 8x8 Numeric Matrix & RGB 3-Channel Tensor Demo
fig, axes = plt.subplots(1, 5, figsize=(14, 3.4))

# 1) Grayscale matrix with exact pixel numbers overlaid
sample_zero = images_raw[0]
axes[0].imshow(sample_zero, cmap="Blues", vmin=0, vmax=16)
axes[0].set_title("Grayscale Matrix (8x8)\nNumbers = Ink (0..16)", fontsize=9.5, fontweight="bold")
for r in range(8):
    for c in range(8):
        val = int(sample_zero[r, c])
        axes[0].text(c, r, str(val), ha="center", va="center",
                     fontsize=7.5, color="white" if val > 8 else "#0f172a", fontweight="bold")
axes[0].set_xticks(range(8))
axes[0].set_yticks(range(8))

# 2..5) Synthetic 8x8x3 RGB Color Image (Cyan-Gold ring on dark background) split into R, G, B matrices
rgb_demo = np.zeros((8, 8, 3), dtype=np.uint8)
rgb_demo[:, :] = [15, 23, 42]                       # Dark slate background
rgb_demo[1:7, 2:6] = [56, 189, 248]                 # Sky-blue outer stroke
rgb_demo[2:6, 3:5] = [250, 204, 21]                 # Warm gold inner core

axes[1].imshow(rgb_demo)
axes[1].set_title("Color RGB Tensor\nShape: (8, 8, 3)", fontsize=9.5, fontweight="bold")
axes[1].axis("off")

channel_titles = ["Red Channel (R)", "Green Channel (G)", "Blue Channel (B)"]
channel_cmaps  = ["Reds", "Greens", "Blues"]
for ch in range(3):
    axes[2 + ch].imshow(rgb_demo[:, :, ch], cmap=channel_cmaps[ch], vmin=0, vmax=255)
    axes[2 + ch].set_title(f"{channel_titles[ch]}\nValues: 0..255", fontsize=9.5, fontweight="bold")
    axes[2 + ch].axis("off")

plt.tight_layout()
plt.show()

---
## 4. Image Preprocessing: Normalization & Stratified Train/Test Split

Before extracting features or measuring distances:
1. **Uniform Intensity Normalization:** Dividing pixel values by `16.0` maps every pixel cleanly from $[0, 16]$ to $[0.0, 1.0]$ while preserving exact relative ink contrast across all $64$ positions.
   - *Important Engineering Note:* Why uniform `X / 16.0` on raw pixels instead of per-pixel `StandardScaler`? Because corner pixels in centered digit grids are almost always `0` ($\sigma \approx 0.05$). Dividing rare stray corner ink by a tiny $\sigma$ magnifies border noise! By contrast, `StandardScaler` is essential when combining engineered features that have **different units or numerical ranges** (which we verify below!).
2. **Stratified Train/Test Split ($75\% / 25\%$):** We hold out **450 unseen test images** (`25%`) so our evaluation genuinely measures generalization—with **zero data leakage**.

In [ ]:
# Normalize pixel intensities uniformly to [0.0, 1.0]
X_norm = X_raw_16 / 16.0
images_norm = images_raw / 16.0

# Stratified 75% Train / 25% Test split
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X_norm, y, images_norm,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"Training set : {X_train.shape[0]} images, feature shape {X_train.shape[1]}-D")
print(f"Testing set  : {X_test.shape[0]} unseen images, feature shape {X_test.shape[1]}-D")
print(f"Normalized intensity range: [{X_train.min():.2f}, {X_train.max():.2f}]")

---
## 5. Feature Extraction: Raw Pixels vs. Handcrafted Features vs. Compact Embeddings

As covered on **Slide 5**, we can represent each image at different levels of sophistication:
1. **Level 1 — Raw Pixel Representation ($64$-D):** Flatten the $8 \times 8$ normalized grid row-by-row into $\mathbf{x} \in \mathbb{R}^{64}$.
2. **Compact Statistical Projection — PCA ($24$-D):** Compress $64$ correlated pixels down to the top $24$ Principal Components (learned **strictly on `X_train`**), discarding empty border variance.
3. **Level 2 — Handcrafted Visual Features ($128$-D):** Human-designed Computer Vision pipeline combining:
   - **Contrast Normalization & Center-of-Mass Alignment** (shifts the ink centroid to the center of the $8 \times 8$ frame so a shifted digit aligns properly!), plus
   - **Mini-HOG (Histogram of Oriented Gradients)** across a $4 \times 4$ grid of $2 \times 2$ pixel cells ($4$ edge orientation bins: $0^\circ, 45^\circ, 90^\circ, 135^\circ$), capturing stroke directions rather than just raw brightness!

In [ ]:
def extract_handcrafted_features(imgs):
    """
    Extracts a 128-D handcrafted feature vector per 8x8 image:
      - 64-D contrast-normalized, center-of-mass aligned intensity grid
      - 64-D Mini-HOG oriented edge gradient histogram (4x4 cells x 4 angle bins)
    """
    features = []
    grid_r, grid_c = np.indices((8, 8))

    for img in imgs:
        # Step 1: Contrast-normalize image to [0, 1] (lighting invariance)
        im = img / (img.max() + 1e-8)

        # Step 2: Align center of mass to grid center (3.5, 3.5) (shift invariance)
        ink_mass = im.sum() + 1e-8
        shift_r = int(np.round(3.5 - (im * grid_r).sum() / ink_mass))
        shift_c = int(np.round(3.5 - (im * grid_c).sum() / ink_mass))

        aligned = np.zeros_like(im)
        for r in range(8):
            for c in range(8):
                nr, nc = r + shift_r, c + shift_c
                if 0 <= nr < 8 and 0 <= nc < 8:
                    aligned[nr, nc] = im[r, c]

        # Step 3: Compute horizontal & vertical edge gradients (gx, gy)
        gx = np.gradient(aligned, axis=1)
        gy = np.gradient(aligned, axis=0)
        magnitude = np.hypot(gx, gy)
        angle_deg = (np.rad2deg(np.arctan2(gy, gx)) + 180.0) % 180.0  # [0, 180)

        # Step 4: 4x4 grid of 2x2 cells x 4 orientation bins = 64 gradient features
        hog_desc = []
        for r in range(0, 8, 2):
            for c in range(0, 8, 2):
                hist, _ = np.histogram(
                    angle_deg[r:r+2, c:c+2],
                    bins=4,
                    range=(0.0, 180.0),
                    weights=magnitude[r:r+2, c:c+2]
                )
                hog_desc.extend(hist)

        hog_desc = np.array(hog_desc)
        hog_desc = hog_desc / (np.linalg.norm(hog_desc) + 1e-8)

        # Concatenate aligned structure + weighted gradient orientation features
        features.append(np.concatenate([aligned.ravel(), hog_desc * 1.5]))

    return np.array(features)

# Extract handcrafted features for train and test sets
X_train_hc = extract_handcrafted_features(img_train)
X_test_hc  = extract_handcrafted_features(img_test)

# Fit compact 24-D PCA strictly on X_train (no data leakage!)
pca_24 = PCA(n_components=24, random_state=42).fit(X_train)
X_train_pca24 = pca_24.transform(X_train)
X_test_pca24  = pca_24.transform(X_test)

print(f"1) Raw Pixel Vector shape        : {X_train.shape[1]}-D")
print(f"2) Compact PCA Vector shape      : {X_train_pca24.shape[1]}-D (retains {pca_24.explained_variance_ratio_.sum()*100:.1f}% variance)")
print(f"3) Handcrafted Feature shape     : {X_train_hc.shape[1]}-D (Center-Aligned + Edge Gradients)")

---
## 6. Feature Vector & Feature Space Visualization

Let's make **Feature Vectors** and **Feature Space** visually concrete (**Slides 5 & 6**):
1. **Top plot:** How a single $8 \times 8$ image becomes a $64$-dimensional feature vector bar chart.
2. **Bottom plot:** Projecting the training images into a **2D PCA Feature Space** so we can literally *see* how images of the same digit cluster **nearby** in geometric space!

In [ ]:
# Top: Single image -> 64-D Feature Vector Bar Chart
fig, (ax_img, ax_vec) = plt.subplots(1, 2, figsize=(12, 3.3), gridspec_kw={"width_ratios": [1, 3.2]})

idx_demo = 0
ax_img.imshow(img_train[idx_demo], cmap="Blues", vmin=0, vmax=1)
ax_img.set_title(f"Training Image #0 (Label: Digit {y_train[idx_demo]})\n8x8 Pixel Matrix", fontsize=10, fontweight="bold")
ax_img.axis("off")

ax_vec.bar(range(64), X_train[idx_demo], color="#2563eb", edgecolor="#1e3a8a", width=0.8)
ax_vec.set_title("Flattened 64-Dimensional Feature Vector  x = [x_0, x_1, ..., x_63]", fontsize=10.5, fontweight="bold")
ax_vec.set_xlabel("Feature Index (Row-by-Row Pixel Coordinate 0..63)")
ax_vec.set_ylabel("Normalized Ink [0..1]")
ax_vec.set_xlim(-1, 64)
plt.tight_layout()
plt.show()

# Bottom: 2D PCA Feature Space Scatter Plot showing nearby class clusters
pca_2d = PCA(n_components=2, random_state=42)
X_train_2d = pca_2d.fit_transform(X_train)

plt.figure(figsize=(9.5, 6.2))
scatter = plt.scatter(
    X_train_2d[:, 0], X_train_2d[:, 1],
    c=y_train, cmap="tab10", alpha=0.78, s=28, edgecolors="white", linewidth=0.3
)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label("True Digit Class (0–9)", fontweight="bold")

# Annotate cluster centroids so clusters are super easy to read
for d_cls in range(10):
    centroid = X_train_2d[y_train == d_cls].mean(axis=0)
    plt.text(
        centroid[0], centroid[1], str(d_cls),
        fontsize=12, fontweight="bold", ha="center", va="center",
        bbox=dict(boxstyle="circle,pad=0.28", facecolor="white", edgecolor="#0f172a", lw=1.5)
    )

plt.title("2D PCA Projection of Feature Space — Visually Similar Digits Cluster Nearby!", fontsize=12, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

---
## 7. Distance & Similarity Measurement: Euclidean vs. Cosine

Let's verify **Slides 7 & 9** mathematically in Python:
1. **Part A — Lecture Worked Example Verification:** Confirm the exact Euclidean distances and Cosine similarities for our 5 training images (`A, B, C, D, E`) and query image $\mathbf{x}_{\text{query}} = [3, 6, 4]$ (`Center Ink`, `Loop Score`, `Vertical Stroke`).
2. **Part B — Real Digit Images Comparison:** Measure Euclidean distance and Cosine similarity between:
   - Two different handwritten `0`s (same class $\rightarrow$ small distance, Cosine $\approx 1$), vs.
   - A handwritten `0` and a handwritten `1` (different class $\rightarrow$ large distance, lower Cosine similarity).

In [ ]:
# Part A: Verify the exact 3D Worked Example from Slide 9 & topic-explanation.md
worked_X = np.array([
    [2.0, 8.0, 3.0],  # Image A -> Digit 0
    [3.0, 7.0, 4.0],  # Image B -> Digit 0
    [8.0, 2.0, 9.0],  # Image C -> Digit 1
    [7.0, 3.0, 8.0],  # Image D -> Digit 1
    [9.0, 1.0, 9.0],  # Image E -> Digit 1
])
worked_y = np.array([0, 0, 1, 1, 1])
worked_ids = ["Image A", "Image B", "Image C", "Image D", "Image E"]
query_vec = np.array([[3.0, 6.0, 4.0]])

euc_dists = euclidean_distances(query_vec, worked_X)[0]
cos_sims  = cosine_similarity(query_vec, worked_X)[0]

print("=== Part A: Slide 9 Worked Example Verification (Query = [3, 6, 4], K = 3) ===")
print(f"{'ID':<10} {'Vector':<16} {'Class':<10} {'Euclidean Dist':<16} {'Cosine Similarity'}")
print("-" * 70)
for name, vec, cls, d_val, c_val in zip(worked_ids, worked_X, worked_y, euc_dists, cos_sims):
    print(f"{name:<10} {str(vec.astype(int).tolist()):<16} Digit {cls:<4} {d_val:<16.4f} {c_val:.4f}")

top3_idx = np.argsort(euc_dists)[:3]
votes = worked_y[top3_idx]
pred_worked = np.bincount(votes).argmax()
print(f"\nTop K=3 Nearest Neighbors : {[worked_ids[i] for i in top3_idx]}")
print(f"Neighbor Labels           : {['Digit ' + str(v) for v in votes]}")
print(f"Majority Vote Prediction  : Digit {pred_worked}\n")

# Part B: Real 64-D Digit Vectors — Same Class ('0' vs '0') vs Different Class ('0' vs '1')
zero_indices = np.where(y_train == 0)[0][:2]
one_index    = np.where(y_train == 1)[0][0]

v_zero_a = X_train[zero_indices[0]].reshape(1, -1)
v_zero_b = X_train[zero_indices[1]].reshape(1, -1)
v_one    = X_train[one_index].reshape(1, -1)

print("=== Part B: Real 64-D Handwritten Digit Vectors ===")
print(f"Digit 0 (#a) vs Digit 0 (#b) -> Euclidean = {euclidean_distances(v_zero_a, v_zero_b)[0,0]:.4f} | Cosine Sim = {cosine_similarity(v_zero_a, v_zero_b)[0,0]:.4f}")
print(f"Digit 0 (#a) vs Digit 1      -> Euclidean = {euclidean_distances(v_zero_a, v_one)[0,0]:.4f} | Cosine Sim = {cosine_similarity(v_zero_a, v_one)[0,0]:.4f}")

---
## 8. K-Nearest Neighbors (KNN) Classification

Now let's initialize and fit `KNeighborsClassifier(n_neighbors=3, metric='euclidean')` on our $1,347$ training images (`X_train`, `y_train`).

Remember: KNN is an **instance-based ("lazy") learner**—calling `.fit()` stores the labeled feature representations in memory so future queries can search their neighborhood! *(And remember: while KNN classifies directly via nearby training points, other vision models like SVMs or CNNs learn parametric decision weights instead.)*

In [ ]:
knn_k3 = KNeighborsClassifier(n_neighbors=3, metric="euclidean")
knn_k3.fit(X_train, y_train)

print("KNN Classifier initialized and fitted!")
print(f"Stored reference representations : {knn_k3.n_samples_fit_} training vectors in R^{knn_k3.n_features_in_}")
print(f"Hyperparameters                  : K = {knn_k3.n_neighbors}, metric = '{knn_k3.metric}'")

---
## 9. Testing with Unseen Images

Let's feed **6 completely unseen test images** into our fitted KNN classifier and inspect both the predicted class and the $K=3$ vote probabilities (`predict_proba`).

In [ ]:
sample_test_indices = [0, 7, 14, 21, 28, 35]
X_sample_test = X_test[sample_test_indices]
y_sample_true = y_test[sample_test_indices]
y_sample_pred = knn_k3.predict(X_sample_test)
y_sample_prob = knn_k3.predict_proba(X_sample_test)

fig, axes = plt.subplots(1, 6, figsize=(13, 2.8))
fig.suptitle("Predictions on Unseen Test Images (K = 3)", fontsize=12, fontweight="bold")

for i, idx in enumerate(sample_test_indices):
    axes[i].imshow(img_test[idx], cmap="Blues", vmin=0, vmax=1)
    conf = y_sample_prob[i, y_sample_pred[i]] * 100
    match_color = "#059669" if y_sample_pred[i] == y_sample_true[i] else "#dc2626"
    axes[i].set_title(
        f"Pred: {y_sample_pred[i]} ({conf:.0f}% vote)\nTrue: {y_sample_true[i]}",
        fontsize=9.5, fontweight="bold", color=match_color
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()

---
## 10. Accuracy Evaluation across All 450 Unseen Test Images

Let's predict all $450$ held-out test images and evaluate overall **Accuracy** alongside per-class **Precision, Recall, and F1-Score**—plus compare Euclidean Distance vs. Cosine Similarity on the same 64-D representations.

In [ ]:
y_pred_euc = knn_k3.predict(X_test)
acc_euc = accuracy_score(y_test, y_pred_euc)

knn_cos = KNeighborsClassifier(n_neighbors=3, metric="cosine").fit(X_train, y_train)
y_pred_cos = knn_cos.predict(X_test)
acc_cos = accuracy_score(y_test, y_pred_cos)

# Also verify the scaling nuance discussed in Slide 7 / topic-explanation.md!
knn_per_pixel_std = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3, metric="euclidean"))
]).fit(X_train, y_train)
acc_std_raw = accuracy_score(y_test, knn_per_pixel_std.predict(X_test))

print(f"Test Accuracy — Uniform [0,1] Pixels + Euclidean (K=3) : {acc_euc*100:.2f}% ({(y_pred_euc == y_test).sum()}/{len(y_test)} correct)")
print(f"Test Accuracy — Uniform [0,1] Pixels + Cosine    (K=3) : {acc_cos*100:.2f}% ({(y_pred_cos == y_test).sum()}/{len(y_test)} correct)")
print(f"Test Accuracy — Per-Pixel StandardScaler (border noise): {acc_std_raw*100:.2f}%\n")

print("Detailed Classification Report (Euclidean, K=3):")
print(classification_report(y_test, y_pred_euc, digits=4))

---
## 11. Confusion Matrix & Error Diagnosis

Accuracy tells us *how often* we succeed (`98.44%`), while the **Confusion Matrix** reveals *which exact visual classes get confused*! Let's plot the $10 \times 10$ confusion matrix and inspect the misclassified test images visually.

In [ ]:
cm = confusion_matrix(y_test, y_pred_euc)

fig, ax = plt.subplots(figsize=(7.2, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Confusion Matrix on 450 Unseen Test Digits (K=3, Accuracy = 98.44%)", fontsize=11.5, fontweight="bold")
plt.tight_layout()
plt.show()

# Visually inspect every misclassified image!
misclassified_idx = np.where(y_pred_euc != y_test)[0]
print(f"Total misclassified test images: {len(misclassified_idx)} out of {len(y_test)}")

fig, axes = plt.subplots(1, len(misclassified_idx), figsize=(2.1 * len(misclassified_idx), 2.8))
fig.suptitle("Inspecting All Misclassified Test Images — Notice AmbiguousStrokes!", fontsize=11.5, fontweight="bold")

for ax, err_i in zip(axes, misclassified_idx):
    ax.imshow(img_test[err_i], cmap="Reds", vmin=0, vmax=1)
    ax.set_title(f"True: {y_test[err_i]} | Pred: {y_pred_euc[err_i]}\n(Test #{err_i})", fontsize=9, fontweight="bold", color="#b91c1c")
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 12. Visualization of Nearest Neighbors

This is the heart of our entire lecture topic: **seeing the exact $K$ nearby training images that determined each classification!**
Below we take 3 unseen test queries (including one tricky misclassified digit `Test #25`!) and plot their **5 nearest neighbors** in `X_train` along with exact Euclidean distances.

In [ ]:
knn_k5_viz = KNeighborsClassifier(n_neighbors=5, metric="euclidean").fit(X_train, y_train)

# Show 2 clean test digits and 1 ambiguous misclassified digit (Test #25: narrow '8' predicted as '1')
showcase_indices = [0, 12, misclassified_idx[0]]
dists, neighbor_indices = knn_k5_viz.kneighbors(X_test[showcase_indices], n_neighbors=5)

fig, axes = plt.subplots(len(showcase_indices), 6, figsize=(12.5, 6.2))
fig.suptitle("Query Test Images (Left) and Their Top-5 Nearest Neighbors in Feature Space", fontsize=12, fontweight="bold")

for row, test_i in enumerate(showcase_indices):
    pred_lbl = knn_k3.predict(X_test[[test_i]])[0]
    true_lbl = y_test[test_i]

    # Column 0: Query image
    axes[row, 0].imshow(img_test[test_i], cmap="Purples", vmin=0, vmax=1)
    status = "OK" if pred_lbl == true_lbl else "ERR"
    axes[row, 0].set_title(f"QUERY #{test_i} [{status}]\nTrue: {true_lbl} | K=3 Pred: {pred_lbl}", fontsize=9, fontweight="bold", color="#5b21b6")
    axes[row, 0].axis("off")

    # Columns 1..5: Nearest training neighbors
    for col in range(5):
        n_idx = neighbor_indices[row, col]
        n_dist = dists[row, col]
        n_lbl = y_train[n_idx]
        axes[row, col + 1].imshow(img_train[n_idx], cmap="Blues", vmin=0, vmax=1)
        axes[row, col + 1].set_title(f"Neighbor #{col+1}\nLabel: {n_lbl} (d={n_dist:.2f})", fontsize=8.8)
        axes[row, col + 1].axis("off")

plt.tight_layout()
plt.show()

---
## 13. Experimenting with Different $K$ Values (Overfitting vs. Underfitting)

How does changing the neighborhood size $K$ affect training accuracy and unseen test accuracy? Let's test $K \in \{1, 3, 5, 7, 9, 11, 15, 21, 25\}$ (**Slide 11**):
- At **$K = 1$**, training accuracy is trivially $100\%$ because every training image is its own closest neighbor ($d = 0$), making boundaries sensitive to local outliers.
- Around **$K = 3$ to $5$**, test accuracy peaks at **$98.44\%$**.
- As **$K$ grows large ($K = 15, 21, 25$)**, distant clusters begin voting and test accuracy steadily drops (**underfitting**).

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 25]
train_scores = []
test_scores_euc = []
test_scores_cos = []

for k in k_values:
    model_e = KNeighborsClassifier(n_neighbors=k, metric="euclidean").fit(X_train, y_train)
    model_c = KNeighborsClassifier(n_neighbors=k, metric="cosine").fit(X_train, y_train)

    train_scores.append(accuracy_score(y_train, model_e.predict(X_train)) * 100)
    test_scores_euc.append(accuracy_score(y_test, model_e.predict(X_test)) * 100)
    test_scores_cos.append(accuracy_score(y_test, model_c.predict(X_test)) * 100)

plt.figure(figsize=(9, 4.5))
plt.plot(k_values, train_scores, "o--", color="#64748b", label="Training Accuracy (Euclidean)")
plt.plot(k_values, test_scores_euc, "s-", linewidth=2.4, color="#2563eb", label="Unseen Test Accuracy (Euclidean)")
plt.plot(k_values, test_scores_cos, "^-", linewidth=2.0, color="#059669", label="Unseen Test Accuracy (Cosine Similarity)")

best_k = k_values[int(np.argmax(test_scores_euc))]
best_acc = max(test_scores_euc)
plt.axvline(3, color="#f59e0b", linestyle=":", linewidth=1.8, label=f"Selected Sweet Spot K=3 ({best_acc:.2f}%)")

plt.title("Effect of Neighborhood Size K on Classification Accuracy", fontsize=12, fontweight="bold")
plt.xlabel("Number of Nearest Neighbors (K)")
plt.ylabel("Accuracy (%)")
plt.xticks(k_values)
plt.ylim(94.5, 100.5)
plt.grid(alpha=0.28)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

for k, tr_s, te_e, te_c in zip(k_values, train_scores, test_scores_euc, test_scores_cos):
    print(f"K = {k:2d} | Train Acc: {tr_s:6.2f}% | Test Acc (Euclidean): {te_e:6.2f}% | Test Acc (Cosine): {te_c:6.2f}%")

---
## 14. Comparison: Raw Pixel Features vs. Improved Image Features

Why did Computer Vision evolve beyond raw pixels toward **Handcrafted Features (HOG/SIFT)** and **Deep CNN Embeddings** (**Slides 12–15**)?

Let's test three representations side-by-side on:
1. **Clean Aligned Test Digits** (standard test set), and
2. **Shifted + Dimmed Test Digits** — where every test image is shifted **just 1 pixel to the right** and dimmed to **$40\%$ brightness**! To human eyes, every digit looks almost identical. Watch what happens to Raw Pixel Euclidean KNN vs. our Center-Aligned + Edge Gradient (Mini-HOG) representation with Cosine Similarity!

In [ ]:
# Train classifiers on each representation
clf_raw = KNeighborsClassifier(n_neighbors=3, metric="euclidean").fit(X_train, y_train)
clf_pca = KNeighborsClassifier(n_neighbors=3, weights="distance", metric="euclidean").fit(X_train_pca24, y_train)
clf_hc  = KNeighborsClassifier(n_neighbors=3, weights="distance", metric="cosine").fit(X_train_hc, y_train)

# Create Shifted (1px right) + Dimmed (40% intensity) test images
img_test_shifted = np.zeros_like(img_test)
img_test_shifted[:, :, 1:] = img_test[:, :, :-1] * 0.40

X_test_shifted_raw   = img_test_shifted.reshape(len(img_test), -1)
X_test_shifted_pca24 = pca_24.transform(X_test_shifted_raw)
X_test_shifted_hc    = extract_handcrafted_features(img_test_shifted)

# Evaluate Clean vs. Shifted+Dimmed
results = {
    "Raw Pixels (64-D, Euc)": (
        accuracy_score(y_test, clf_raw.predict(X_test)) * 100,
        accuracy_score(y_test, clf_raw.predict(X_test_shifted_raw)) * 100,
    ),
    "Compact PCA (24-D, Euc)": (
        accuracy_score(y_test, clf_pca.predict(X_test_pca24)) * 100,
        accuracy_score(y_test, clf_pca.predict(X_test_shifted_pca24)) * 100,
    ),
    "Handcrafted Aligned+HOG (128-D, Cos)": (
        accuracy_score(y_test, clf_hc.predict(X_test_hc)) * 100,
        accuracy_score(y_test, clf_hc.predict(X_test_shifted_hc)) * 100,
    ),
}

print(f"{'Representation':<38} {'Clean Test Acc':<18} {'1px Shift + 40% Brightness Acc'}")
print("-" * 88)
for name, (clean_acc, shift_acc) in results.items():
    print(f"{name:<38} {clean_acc:>10.2f}%        {shift_acc:>14.2f}%")

# Visualize the comparison!
fig, (ax_demo, ax_bar) = plt.subplots(1, 2, figsize=(12.5, 4.3), gridspec_kw={"width_ratios": [1, 2.2]})

# Show clean vs shifted+dimmed sample image side by side
combined_preview = np.hstack([img_test[0], np.ones((8, 1)) * 0.5, img_test_shifted[0]])
ax_demo.imshow(combined_preview, cmap="Blues", vmin=0, vmax=1)
ax_demo.set_title("Left: Clean Test '0' | Right: +1px Right & 40% Dim\n(Humans easily read both as '0'!)", fontsize=9.5, fontweight="bold")
ax_demo.axis("off")

labels = list(results.keys())
clean_vals = [results[k][0] for k in labels]
shift_vals = [results[k][1] for k in labels]

x_pos = np.arange(len(labels))
width = 0.35
bars1 = ax_bar.bar(x_pos - width/2, clean_vals, width, label="Clean Aligned Test Set", color="#2563eb")
bars2 = ax_bar.bar(x_pos + width/2, shift_vals, width, label="1-Pixel Shift + 40% Dimmed Test Set", color="#f59e0b")

ax_bar.set_ylabel("Classification Accuracy (%)")
ax_bar.set_title("Why Representation Quality Matters: Raw Pixels Crash Under Small Shifts!", fontsize=11, fontweight="bold")
ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(["Raw Pixels\n(64-D)", "Compact PCA\n(24-D)", "Aligned + Edge HOG\n(128-D + Cosine)"])
ax_bar.set_ylim(0, 112)
ax_bar.legend(loc="lower right")

for b in list(bars1) + list(bars2):
    ax_bar.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5, f"{b.get_height():.1f}%",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 15. Test Your Own Custom Image (Interactive Upload + Auto-Demo)

Want to test **your own handwritten digit photo or drawing** during a live Google Colab presentation?
- **Automatic Out-of-the-Box Demo (`UPLOAD_CUSTOM_FILE = False`):** Runs cleanly during `Runtime -> Run all` using a simulated high-resolution $64 \times 64$ hand-drawn digit canvas so the notebook never hangs waiting for a dialog.
- **Live File Upload in Google Colab (`UPLOAD_CUSTOM_FILE = True`):** Simply change `UPLOAD_CUSTOM_FILE = True` and run the cell below! A file upload button will appear—upload any PNG/JPG image of a single digit (`0–9`, dark ink on light paper or white ink on dark background), and our pipeline will crop, resize to $8 \times 8$, extract features, predict its class, and plot its $K = 5$ nearest neighbors!

In [ ]:
def preprocess_custom_image_to_8x8(pil_img):
    """
    Converts an arbitrary RGB/Grayscale PIL image of a single digit into an 8x8
    normalized [0, 1] ink array matching sklearn's load_digits convention.
    """
    gray = pil_img.convert("L")
    arr = np.array(gray, dtype=np.float32)

    # Auto-invert if background is bright (e.g. dark pen on white paper)
    border_pixels = np.concatenate([arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]])
    if np.median(border_pixels) > 127.0:
        arr = 255.0 - arr

    # Remove faint background illumination noise
    arr = np.clip(arr - np.percentile(arr, 15), 0, None)
    if arr.max() > 0:
        arr = arr / arr.max()

    # Tight bounding-box crop around the drawn digit + square padding
    ys, xs = np.where(arr > 0.18)
    if len(ys) > 0 and len(xs) > 0:
        r0, r1 = ys.min(), ys.max() + 1
        c0, c1 = xs.min(), xs.max() + 1
        cropped = arr[r0:r1, c0:c1]
        side = max(cropped.shape[0], cropped.shape[1])
        pad_px = max(2, int(side * 0.22))
        canvas = np.zeros((side + 2 * pad_px, side + 2 * pad_px), dtype=np.float32)
        y_off = (canvas.shape[0] - cropped.shape[0]) // 2
        x_off = (canvas.shape[1] - cropped.shape[1]) // 2
        canvas[y_off:y_off + cropped.shape[0], x_off:x_off + cropped.shape[1]] = cropped
    else:
        canvas = arr

    # Smoothly downsample to 8x8 using PIL LANCZOS resampling
    canvas_uint8 = (canvas * 255.0).astype(np.uint8)
    resized_8x8 = np.array(Image.fromarray(canvas_uint8).resize((8, 8), Image.Resampling.LANCZOS), dtype=np.float32)
    if resized_8x8.max() > 0:
        resized_8x8 = resized_8x8 / resized_8x8.max()
    return resized_8x8

def build_synthetic_handdrawn_zero():
    """Creates a realistic 64x64 high-res 'hand-drawn digit 0' on white paper for instant Run-All demo."""
    img_pil = Image.new("L", (64, 64), color=248)
    draw = ImageDraw.Draw(img_pil)
    # Draw a natural slightly-oval thick pen loop on white paper
    draw.ellipse([16, 8, 48, 56], outline=16, width=8)
    return img_pil

# ==============================================================================
# Toggle UPLOAD_CUSTOM_FILE = True in Google Colab to upload your own PNG/JPG!
# ==============================================================================
UPLOAD_CUSTOM_FILE = False

custom_pil = None
custom_source_label = "Simulated 64x64 Hand-Drawn Digit '0' (High-Res Canvas)"

if UPLOAD_CUSTOM_FILE:
    try:
        from google.colab import files
        print("Please choose an image file (PNG/JPG) of a single digit (0-9)...")
        uploaded = files.upload()
        if uploaded:
            fname = next(iter(uploaded.keys()))
            custom_pil = Image.open(fname)
            custom_source_label = f"Uploaded File: {fname}"
    except ImportError:
        print("Not running inside Google Colab; falling back to built-in custom canvas image.")

if custom_pil is None:
    custom_pil = build_synthetic_handdrawn_zero()

# Preprocess to 8x8, extract features, classify, and inspect Top-5 Neighbors!
custom_8x8 = preprocess_custom_image_to_8x8(custom_pil)
custom_feat_hc = extract_handcrafted_features(custom_8x8[np.newaxis, :, :])

pred_digit = clf_hc.predict(custom_feat_hc)[0]
nn_dists, nn_idxs = clf_hc.kneighbors(custom_feat_hc, n_neighbors=5)

print(f"Input Source             : {custom_source_label}")
print(f"Predicted Digit Class    : Digit {pred_digit}")
print(f"Top-5 Neighbor Labels    : {y_train[nn_idxs[0]].tolist()}")
print(f"Top-5 Cosine Distances   : {[round(float(d), 4) for d in nn_dists[0]]}")

fig, axes = plt.subplots(1, 7, figsize=(13.5, 2.5))
fig.suptitle(f"Custom Image Classification -> Predicted: Digit {pred_digit}", fontsize=12, fontweight="bold")

axes[0].imshow(custom_pil, cmap="gray")
axes[0].set_title("1. Original Input\nImage", fontsize=8.8, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(custom_8x8, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title(f"2. Preprocessed\n8x8 Grid (Pred: {pred_digit})", fontsize=8.8, fontweight="bold", color="#059669")
axes[1].axis("off")

for i in range(5):
    t_idx = nn_idxs[0, i]
    axes[2 + i].imshow(img_train[t_idx], cmap="Blues", vmin=0, vmax=1)
    axes[2 + i].set_title(f"Neighbor #{i+1}\nClass {y_train[t_idx]} (d={nn_dists[0, i]:.2f})", fontsize=8.5)
    axes[2 + i].axis("off")

plt.tight_layout()
plt.show()

---
## 16. Summary & Key Conclusions

1. **Images Are Numerical Tensors:** Grayscale images are 2D matrices ($H \times W$) and color images are 3D RGB tensors ($H \times W \times 3$). Feature extraction transforms each image into a coordinate point $\mathbf{x} \in \mathbb{R}^d$ in **Feature Space**.
2. **Nearby Feature Representations Share Classes:** Using **Euclidean Distance** ($L_2$) or **Cosine Similarity**, K-Nearest Neighbors (KNN) classifies an unseen image by taking a majority vote among its $K$ closest labeled neighbors—achieving **$98.44\%$ accuracy** ($K=3$) on unseen handwritten digits!
3. **Hyperparameter $K$ Balances Noise vs. Smoothing:** Very small $K$ ($K=1$) can overfit single noisy samples, whereas overly large $K$ ($K=21+$) underfits by pulling votes from neighboring classes.
4. **Representation Quality Is Everything:** Raw pixel vectors collapsed from **`98.44%` to `52.00%`** under a mere **1-pixel shift + dimmed lighting**, whereas **Center-Aligned + Oriented Edge Gradient (Mini-HOG)** features with **Cosine Similarity** held steady at **`98.00%`**!
5. **Bridge to Modern Deep Learning (CNNs):** While not every vision system uses KNN as its final classifier, modern **Convolutional Neural Networks (CNNs)** and **Vision Transformers** learn compact $128$-D to $512$-D **feature embeddings** where semantic similarity corresponds directly to geometric closeness—powering Face ID, Google Lens visual search, medical case retrieval, and billion-scale vector databases!